In [1]:
from dotenv import load_dotenv
load_dotenv()

from ragwire import RAGWire, setup_logging
import ragwire
logger = setup_logging(log_level="INFO")

print(ragwire.__version__)

rag = RAGWire("config_gemini.yaml")
# rag = RAGWire("config_openai.yaml")
# rag = RAGWire("config_groq.yaml")

c:\Users\laxmi\anaconda3\envs\ml\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


1.2.7
2026-03-26 19:57:23,253 - ragwire.core.pipeline - INFO - Loading configuration from config_gemini.yaml
2026-03-26 19:57:23,610 - ragwire.core.pipeline - INFO - Document loader initialized
2026-03-26 19:57:23,611 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)
2026-03-26 19:57:24,700 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=google)
2026-03-26 19:57:24,704 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=google, model=gemini-3.1-flash-lite-preview)
2026-03-26 19:57:25,087 - ragwire.vectorstores.qdrant_store - INFO - Using local Qdrant storage at ./qdrant_storage
2026-03-26 19:57:25,615 - ragwire.vectorstores.qdrant_store - INFO - Created collection 'my_docs' with dense vectors only
2026-03-26 19:57:25,616 - ragwire.core.pipeline - INFO - Created new collection: my_docs
2026-03-26 19:57:26,093 - ragwire.core.pipeline - INFO - Vector store initialized
2026-03-26 19:57:26,0

In [2]:
stats = rag.ingest_directory('../data')

2026-03-26 19:57:35,973 - ragwire.core.pipeline - INFO - Found 3 file(s) in ../data
2026-03-26 19:57:35,974 - ragwire.core.pipeline - INFO - Starting ingestion of 3 documents


Ingesting:   0%|          | 0/3 [00:00<?, ?file/s]

2026-03-26 19:58:18,403 - ragwire.core.pipeline - INFO - Processed ..\data\amazon 10k 2025.pdf: 38 chunks


Ingesting:  33%|███▎      | 1/3 [00:42<01:24, 42.43s/file]

2026-03-26 19:58:48,626 - ragwire.core.pipeline - INFO - Processed ..\data\Apple_10k_2025.pdf: 35 chunks


Ingesting:  67%|██████▋   | 2/3 [01:12<00:35, 35.25s/file]

2026-03-26 19:59:29,304 - ragwire.core.pipeline - INFO - Processed ..\data\GOOG-10-K-2025.pdf: 46 chunks


Ingesting: 100%|██████████| 3/3 [01:53<00:00, 37.78s/file]

2026-03-26 19:59:29,307 - ragwire.core.pipeline - INFO - Ingestion complete: 3/3 documents


In [3]:
from typing import Optional

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

# ------------------------------------------------------------------ #
# 2. Tools
# ------------------------------------------------------------------ #
@tool
def get_filter_context(query: str) -> str:
    """Get available metadata fields, stored values, and filter suggestions for a query.

    Call this before search_documents when the query involves specific metadata
    (company, year, document type, etc.). Use the returned context to decide
    what filters to pass to search_documents.

    Skip this for purely semantic queries with no metadata intent.
    """
    return rag.get_filter_context(query)


@tool
def search_documents(query: str, filters: Optional[dict] = None) -> str:
    """Search the document knowledge base for relevant information.

    Args:
        query: The search query
        filters: Optional metadata filters decided from get_filter_context.
                 Pass {} or omit to search without filtering.
    """
    results = rag.retrieve(query, top_k=1, filters=filters)
    if not results:
        return "No relevant documents found."

    chunks = []
    for doc in results:
        source = doc.metadata.get("file_name", "unknown")
        meta_parts = [
            f"{k}={str(v)[:100]}"
            for k, v in doc.metadata.items()
            if k != "file_name" and v not in (None, "", [])
        ]
        header = f"[{source}" + (f" | {', '.join(meta_parts)}" if meta_parts else "") + "]"
        chunks.append(f"{header}\n{doc.page_content}")

    return "\n\n---\n\n".join(chunks)


# ------------------------------------------------------------------ #
# 3. Agent with memory
# ------------------------------------------------------------------ #
# model = ChatOllama(model="qwen3.5:27b", base_url="http://localhost:11434")
model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite-preview')
# model = ChatOpenAI(model='gpt-5.4-nano')
# model = ChatGroq(model='qwen/qwen3-32b')
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_filter_context, search_documents],
    system_prompt=(
        "You are a helpful financial document assistant. "
        "For complex questions, break them down into simple sub-questions and answer each one before forming a final answer. "
        "Always use search_documents to retrieve information before answering — never answer from general knowledge. "
        "Use get_filter_context before search_documents when the query involves specific metadata (company, year, document type, etc.). "
        "If no relevant documents are found, say so — do not guess or fabricate an answer. "
        "Always cite the source document in your answer."
    ),
    checkpointer=checkpointer,
)



config = {"configurable": {"thread_id": "demo-1"}}


# ------------------------------------------------------------------ #
# 4. Interactive Q&A loop
# ------------------------------------------------------------------ #
print("\nRAG Agent ready. Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        break
    if not question:
        continue

    response = agent.invoke(
        {"messages": [HumanMessage(question)]},
        config=config,
    )
    print(f"\nAgent: {response['messages'][-1].text}\n\n\n")



RAG Agent ready. Type 'quit' to exit.

2026-03-26 19:59:59,480 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'apple inc.'}
2026-03-26 20:00:06,857 - ragwire.core.pipeline - INFO - Retrieved 1 documents for query: revenue for Apple Inc. 2025...

Agent: According to Apple Inc.'s 2025 Form 10-K, the company's total net sales for the fiscal year 2025 were $416.161 billion.

Source: Apple Inc. 2025 Form 10-K, "Segment Operating Performance" and "Products and Services Performance" tables.



